In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut

# ============================================================
# PATHS
# ============================================================
VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

# ============================================================
# LOAD DATA
# ============================================================
vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R   = np.load(NEURAL_PATH).T                                   # (images, neurons)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================
top1 = np.argmax(vit, axis=1)
y_true = (top1 <= 397).astype(int)
print("Animate fraction:", y_true.mean())

# ============================================================
# STANDARDIZE NEURAL FEATURES
# ============================================================
scaler = StandardScaler()
X = scaler.fit_transform(R)

# ============================================================
# LOO ACCURACY FUNCTION
# ============================================================
def loo_logistic_accuracy(X, y):
    loo = LeaveOneOut()
    correct = 0

    for train_idx, test_idx in loo.split(X):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx]

        clf = LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2000,
        )
        clf.fit(Xtr, ytr)
        yhat = clf.predict(Xte)

        correct += int(yhat[0] == yte[0])

    return correct / len(y)

# ============================================================
# TRUE LABEL PERFORMANCE
# ============================================================
acc_true = loo_logistic_accuracy(X, y_true)
print("\n=== TRUE LABELS ===")
print("LOO accuracy:", acc_true)

# ============================================================
# PERMUTATION NULL
# ============================================================
n_perm = 500
rng = np.random.default_rng(0)

acc_null = np.zeros(n_perm)

for k in range(n_perm):
    y_perm = rng.permutation(y_true)
    acc_null[k] = loo_logistic_accuracy(X, y_perm)

# ============================================================
# SIGNIFICANCE
# ============================================================
p_value = (1 + np.sum(acc_null >= acc_true)) / (n_perm + 1)

print("\n=== PERMUTATION NULL ===")
print("Null mean / median / max:",
      acc_null.mean(),
      np.median(acc_null),
      acc_null.max())

print("\n=== SIGNIFICANCE ===")
print("Permutation p-value:", p_value)
print("Effect size (true / null median):", acc_true / np.median(acc_null))


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458

=== TRUE LABELS ===
LOO accuracy: 0.6610169491525424

=== PERMUTATION NULL ===
Null mean / median / max: 0.5342372881355932 0.5254237288135594 0.7033898305084746

=== SIGNIFICANCE ===
Permutation p-value: 0.02594810379241517
Effect size (true / null median): 1.2580645161290323
